In [1]:
import os
import sys
import pandas as pd

In [2]:
# Add the root directory to our system path so we can import from 'src' seamlessly
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.preprocess import clean_data

# Load the raw data from your data/raw directory
# (Since the notebook is inside 'notebooks/', we navigate up '..' and then down into 'data/raw/')
raw_data_path = os.path.join('..', 'data', 'raw', 'diabetic_data.csv')
df_raw_load = pd.read_csv(raw_data_path)

# Test your function!
df_processed = clean_data(df_raw_load)

# Print validation metrics to check if Step 1 works perfectly
print(f"Processed Dataframe Shape: {df_processed.shape}")
print(f"Max Medications Check (Should be capped at 60): {df_processed['num_medications'].max()}")
print(f"Unknown Specialty Count: {df_processed['medical_specialty'].value_counts().get('Unknown', 0)}")

Processed Dataframe Shape: (69883, 45)
Max Medications Check (Should be capped at 60): 60
Unknown Specialty Count: 33561


In [4]:
df_processed.columns

Index(['race', 'gender', 'age', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'medical_specialty', 'num_lab_procedures', 'num_procedures',
       'num_medications', 'number_outpatient', 'number_emergency',
       'number_inpatient', 'number_diagnoses', 'max_glu_serum', 'A1Cresult',
       'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
       'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'target',
       'diag_1_clean', 'diag_2_clean', 'diag_3_clean', 'total_prior_visits'],
      dtype='object')

In [11]:
df_processed['diabetesMed'].value_counts()

diabetesMed
Yes    53222
No     16661
Name: count, dtype: int64

In [12]:
# 1. Define explicit columns lists
numerical_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures', 
    'num_medications', 'number_outpatient', 'number_emergency', 
    'number_inpatient', 'number_diagnoses', 'total_prior_visits'
]

# All columns that contain text labels that need One-Hot Encoding
categorical_cols = [
    'race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 
    'admission_source_id', 'medical_specialty', 'max_glu_serum', 'A1Cresult', 
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 
    'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 
    'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 
    'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 
    'metformin-rosiglitazone', 'metformin-pioglitazone',
    'diag_1_clean', 'diag_2_clean', 'diag_3_clean'
]

# Columns that can be passed directly into the model once converted to 0 and 1
passthrough_cols = ['change_binary', 'diabetesMed_binary']

# 2. Convert string binary flags into clean 0/1 passthrough integers before building the transformer
df_processed['change_binary'] = (df_processed['change'] == 'Ch').astype(int)
df_processed['diabetesMed_binary'] = (df_processed['diabetesMed'] == 'Yes').astype(int)

# 3. Create our final clean feature matrix X and target array y
# We explicitly select ONLY the features in our defined lists to guarantee no feature leakage
feature_cols = numerical_cols + categorical_cols + passthrough_cols
X = df_processed[feature_cols]
y = df_processed['target']

print(f"Total features selected for modeling: {len(feature_cols)}")
print(f"Target column distribution:\n{y.value_counts()}")

Total features selected for modeling: 44
Target column distribution:
target
0    63638
1     6245
Name: count, dtype: int64


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Define the separate pipeline blocks
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 2. Combine all blocks into a unified ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numerical_cols),
        ('cat', cat_pipeline, categorical_cols),
        ('pass', 'passthrough', passthrough_cols)
    ],
    remainder='drop' # Automatically drops anything not listed, preventing feature leakage
)

print("ColumnTransformer constructed successfully!")
print(preprocessor)

ColumnTransformer constructed successfully!
ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['time_in_hospital', 'num_lab_procedures',
                                  'num_procedures', 'num_medications',
                                  'number_outpatient', 'number_emergency',
                                  'number_inpatient', 'number_diagnoses',
                                  'total_prior_visits']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImpu...
                                  'chlorpropamide', 'glimepiride',
                                  'acetohexamide', 'glipizide', 'glyburide',
                          

In [14]:
from sklearn.model_selection import train_test_split

# Perform the stratified train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

print("Train/Test Split complete!")
print(f"Original Training Features Shape: {X_train.shape}")
print(f"Original Testing Features Shape: {X_test.shape}")

Train/Test Split complete!
Original Training Features Shape: (55906, 44)
Original Testing Features Shape: (13977, 44)


In [15]:
from imblearn.over_sampling import SMOTE

# 1. Fit the transformer on X_train and transform it
X_train_transformed = preprocessor.fit_transform(X_train)

# 2. Transform X_test using the already fitted transformers (No fitting here!)
X_test_transformed = preprocessor.transform(X_test)

# 3. Instantiate SMOTE and resample the training set only
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_transformed, y_train)

print("Preprocessing transformations and SMOTE oversampling complete!")

Preprocessing transformations and SMOTE oversampling complete!


In [30]:
# Get feature names from the fitted ColumnTransformer
feature_names = preprocessor.get_feature_names_out()

# Wrap SMOTE output back into DataFrames with proper column names
X_train_resampled = pd.DataFrame(X_train_resampled, columns=feature_names)
X_test_transformed = pd.DataFrame(X_test_transformed, columns=feature_names)

print(f"Feature names restored: {len(feature_names)} columns")
print(f"X_train_resampled type: {type(X_train_resampled)}")
print(f"X_test_transformed type: {type(X_test_transformed)}")

Feature names restored: 246 columns
X_train_resampled type: <class 'pandas.core.frame.DataFrame'>
X_test_transformed type: <class 'pandas.core.frame.DataFrame'>


In [16]:
print("=== TRAINING SET DISTRIBUTION (AFTER SMOTE) ===")
print("Value Counts:")
print(y_train_resampled.value_counts())
print("\nPercentage Breakdown:")
print(y_train_resampled.value_counts(normalize=True) * 100)
print(f"X_train_resampled shape: {X_train_resampled.shape}")

print("\n")

print("=== TESTING SET DISTRIBUTION (UNTOUCHED REAL-WORLD) ===")
print("Value Counts:")
print(y_test.value_counts())
print("\nPercentage Breakdown:")
print(y_test.value_counts(normalize=True) * 100)
print(f"X_test_transformed shape: {X_test_transformed.shape}")

=== TRAINING SET DISTRIBUTION (AFTER SMOTE) ===
Value Counts:
target
0    50910
1    50910
Name: count, dtype: int64

Percentage Breakdown:
target
0    50.0
1    50.0
Name: proportion, dtype: float64
X_train_resampled shape: (101820, 246)


=== TESTING SET DISTRIBUTION (UNTOUCHED REAL-WORLD) ===
Value Counts:
target
0    12728
1     1249
Name: count, dtype: int64

Percentage Breakdown:
target
0    91.063891
1     8.936109
Name: proportion, dtype: float64
X_test_transformed shape: (13977, 246)


MLflow experiment

In [17]:
import mlflow

EXPERIMENT_NAME = "hospital-readmission-prediction"

mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking is initialized.")
print(f"Active Experiment Workspace: '{EXPERIMENT_NAME}'")

c:\Users\aadhi\Downloads\Hospital Readmission Risk Predictor\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/06/13 20:41:43 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/13 20:41:43 INFO mlflow.store.db.utils: Updating database tables
2026/06/13 20:41:48 INFO mlflow.tracking.fluent: Experiment with name 'hospital-readmission-prediction' does not exist. Creating a new experiment.


MLflow tracking is initialized.
Active Experiment Workspace: 'hospital-readmission-prediction'


In [31]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced'      # handles imbalance natively
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced',     # handles imbalance natively
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        random_state=42,
        eval_metric='logloss',
        scale_pos_weight=9,          # ratio of negatives to positives
        use_label_encoder=False
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=100,
        random_state=42,
        is_unbalance=True,           # tells LightGBM about imbalance
        scale_pos_weight=9,
        verbose=-1
    )
}

In [32]:
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score,
    precision_score, accuracy_score
)

def evaluate_at_threshold(y_true, y_proba, threshold):
    y_pred = (y_proba >= threshold).astype(int)
    return {
        "auc_roc":   roc_auc_score(y_true, y_proba),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "accuracy":  accuracy_score(y_true, y_pred)
    }

In [33]:
import mlflow
import mlflow.sklearn
import numpy as np

THRESHOLD = 0.3   # clinical setting — catching more positives matters more
results = {}

for model_name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")

    with mlflow.start_run(run_name=model_name):

        # --- TRAIN ---
        model.fit(X_train_resampled, y_train_resampled)

        # --- PREDICT ---
        y_proba = model.predict_proba(X_test_transformed)[:, 1]

        # --- EVALUATE at multiple thresholds ---
        metrics_05 = evaluate_at_threshold(y_test, y_proba, threshold=0.5)
        metrics_03 = evaluate_at_threshold(y_test, y_proba, threshold=0.3)
        metrics_02 = evaluate_at_threshold(y_test, y_proba, threshold=0.2)

        # --- LOG PARAMS ---
        mlflow.log_param("model_name", model_name)
        mlflow.log_param("threshold_primary", THRESHOLD)
        mlflow.log_param("smote_applied", True)
        mlflow.log_param("train_size", len(X_train_resampled))
        mlflow.log_param("test_size", len(X_test_transformed))

        # --- LOG METRICS (primary threshold 0.3) ---
        mlflow.log_metric("auc_roc",   metrics_03["auc_roc"])
        mlflow.log_metric("f1",        metrics_03["f1"])
        mlflow.log_metric("recall",    metrics_03["recall"])
        mlflow.log_metric("precision", metrics_03["precision"])
        mlflow.log_metric("accuracy",  metrics_03["accuracy"])

        # --- LOG METRICS at other thresholds too ---
        for metric_name, value in metrics_05.items():
            mlflow.log_metric(f"{metric_name}_at_05", value)
        for metric_name, value in metrics_02.items():
            mlflow.log_metric(f"{metric_name}_at_02", value)

        # --- LOG MODEL ---
        mlflow.sklearn.log_model(model, artifact_path="model")

        # --- STORE for comparison table ---
        results[model_name] = {
            "AUC-ROC":   metrics_03["auc_roc"],
            "F1":        metrics_03["f1"],
            "Recall":    metrics_03["recall"],
            "Precision": metrics_03["precision"],
            "Accuracy":  metrics_03["accuracy"],
            "Prob Mean": round(y_proba.mean(), 4),
            "Prob Max":  round(y_proba.max(), 4)
        }

        print(f"  AUC-ROC:   {metrics_03['auc_roc']:.4f}")
        print(f"  Recall:    {metrics_03['recall']:.4f}")
        print(f"  F1:        {metrics_03['f1']:.4f}")
        print(f"  Precision: {metrics_03['precision']:.4f}")
        print(f"  Prob Mean: {y_proba.mean():.4f} | Prob Max: {y_proba.max():.4f}")


Training: Logistic Regression


2026/06/13 21:12:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/13 21:12:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


  AUC-ROC:   0.6381
  Recall:    0.9311
  F1:        0.1764
  Precision: 0.0974
  Prob Mean: 0.4577 | Prob Max: 0.9941

Training: Random Forest


2026/06/13 21:14:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/13 21:14:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


OSError: [Errno 28] No space left on device

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("AUC-ROC", ascending=False)

print("\n" + "="*70)
print("MODEL COMPARISON (at threshold 0.3)")
print("="*70)
print(results_df.to_string())